# 13. H=3 Rolling Propensity Redesign — Top-50 Baselines, Warning Audits, EDA FE (~150) & Final Model (Colab GPU)

- **Problem Definition:** `H = 3`, `snapshot = 1`, strict Moscow cutoff `< YYYY-MM-01 00:00:00 MSK`, full history before cutoff.
- **Zero OOT Leakage:** OOT Test period is `2022-11..2023-01` (evaluated at `2022-11`). Months `2022-09..2023-01` are strictly quarantined and never appear in any train or validation split (`Val: 2022-05..2022-08`, `Test: 2022-11`).
- **5-Fold Disjoint-Client OOC:** Every step trains on `fold != k`, evaluates on `fold == k`, with GPU acceleration (`XGBoost CUDA`, `CatBoost GPU`) and `< 30m` per-model guardrails.

In [ ]:
import sys, subprocess
from pathlib import Path

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")

!nvidia-smi || true
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "catboost", "lightgbm", "xgboost", "scikit-learn", "pandas", "pyarrow", "scipy", "matplotlib"
])

In [ ]:
%%bash
set -e
mkdir -p /content/mbd_work/outputs/h3_propensity/delta /content/drive/MyDrive/mbd_h3_outputs
if [ ! -f /content/mbd_work/outputs/baseline_moscow_v1/panel_moscow.parquet ]; then
  unzip -q -o /content/drive/MyDrive/mbd_colab_bundle.zip -d /content/mbd_work
fi
if [ ! -f /content/mbd_work/outputs/fe_v2/panel_fe_v2_candidates.parquet ]; then
  unzip -q -o /content/drive/MyDrive/mbd_colab_patch_core.zip -d /content/mbd_work
fi
rm -rf /tmp/prop_repo && git clone --depth 1 https://github.com/AnhQun18/Propensity.git /tmp/prop_repo
cp -r /tmp/prop_repo/src /tmp/prop_repo/configs /tmp/prop_repo/tests /content/mbd_work/
cd /content/mbd_work && python -B -X utf8 -m unittest tests/test_h3_leakage_guards.py -v

In [ ]:
# Stage 1: 5 Top-50 Baselines + 6 Project Warning Verifications
%cd /content/mbd_work
!python -B -X utf8 src/scripts/run_h3_colab_stage1_baselines_and_verify.py \
  --work-dir /content/mbd_work \
  --delta-zip /content/h3_stage1_delta.zip \
  --drive-save-dir /content/drive/MyDrive/mbd_h3_outputs/stage1 \
  --seeds 42 142 242

In [ ]:
# Stage 2: EDA FE (~260+ candidates) -> Val-only Selection (~150) -> GPU Tuning -> Final Ensemble & OOT Test
%cd /content/mbd_work
!python -B -X utf8 src/scripts/run_h3_colab_stage2_fe150_and_final.py \
  --work-dir /content/mbd_work \
  --stage1-delta-zip /content/h3_stage1_delta.zip \
  --stage2-delta-zip /content/h3_stage2_fe_delta.zip \
  --drive-save-dir /content/drive/MyDrive/mbd_h3_outputs/stage2 \
  --seeds 42 142 242
!zip -q -j /content/h3_results_summary.zip outputs/h3_propensity/stage1/*.csv outputs/h3_propensity/stage2/*.csv outputs/h3_propensity/stage2/*.json
!cp -f /content/h3_results_summary.zip /content/drive/MyDrive/mbd_h3_outputs/h3_results_summary.zip
!ls -lh /content/h3_results_summary.zip